# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [57]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [58]:
import requests, os
from urllib.parse import urlparse

url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/zillow_dataset.csv"
filename = os.path.basename(urlparse(url).path)

response = requests.get(url)
with open(filename, "wb") as f:
    f.write(response.content)
print("Downloaded!")

df = pd.read_csv(filename)
df.head()

Downloaded!


,parcelid,airconditioningtypeid,architecturalstyletypeid,basementsqft,bathroomcnt,bedroomcnt,buildingclasstypeid,buildingqualitytypeid,calculatedbathnbr,decktypeid,...,yardbuildingsqft17,yardbuildingsqft26,yearbuilt,numberofstories,fireplaceflag,assessmentyear,taxdelinquencyflag,taxdelinquencyyear,censustractandblock,taxvaluedollarcnt
0,14297519,NaN,NaN,NaN,3.5,4.0,NaN,NaN,3.5,NaN,...,NaN,NaN,1998.0,NaN,NaN,2016.0,NaN,NaN,6.059063e+13,1023282.0
1,17052889,NaN,NaN,NaN,1.0,2.0,NaN,NaN,1.0,NaN,...,NaN,NaN,1967.0,1.0,NaN,2016.0,NaN,NaN,6.111001e+13,464000.0
2,14186244,NaN,NaN,NaN,2.0,3.0,NaN,NaN,2.0,NaN,...,NaN,NaN,1962.0,1.0,NaN,2016.0,NaN,NaN,6.059022e+13,564778.0
3,12177905,NaN,NaN,NaN,3.0,4.0,NaN,8.0,3.0,NaN,...,NaN,NaN,1970.0,NaN,NaN,2016.0,NaN,NaN,6.037300e+13,145143.0
4,10887214,1.0,NaN,NaN,3.0,3.0,NaN,8.0,3.0,NaN,...,NaN,NaN,1964.0,NaN,NaN,2016.0,NaN,NaN,6.037124e+13,119407.0


In [59]:

df_cleaned = df.dropna(thresh=len(df)*0.5, axis=1)  # drop cols with >50% missing
df_cleaned = df_cleaned.dropna()                      # drop remaining rows with nulls
df_cleaned = df_cleaned.select_dtypes(include=[np.number])  # numeric only

X = df_cleaned.drop(columns=["taxvaluedollarcnt"])
y = df_cleaned["taxvaluedollarcnt"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Features: {list(X.columns)}")

Train: (35154, 23), Test: (8789, 23)
Features: ['parcelid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'fips', 'fullbathcnt', 'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet', 'propertylandusetypeid', 'rawcensustractandblock', 'regionidcity', 'regionidcounty', 'regionidzip', 'roomcnt', 'unitcnt', 'yearbuilt', 'assessmentyear', 'censustractandblock']


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [60]:
# Add as many cells as you need
from sklearn.model_selection import RepeatedKFold, cross_val_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso()
}

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, 
                             scoring="neg_mean_absolute_error", cv=cv)
    mae_scores = -scores
    print(f"{name}: MAE = {mae_scores.mean():.2f} +/- {mae_scores.std():.2f}")

Linear Regression: MAE = 249625.67 +/- 3055.23
Ridge Regression: MAE = 249619.64 +/- 3056.32


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.913e+14, tolerance: 1.043e+12
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.306e+14, tolerance: 9.744e+11
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the featu

Lasso Regression: MAE = 249623.80 +/- 3055.60


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.356e+14, tolerance: 1.071e+12
  model = cd_fast.enet_coordinate_descent(


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

> Your text here

All three models performed similarly, with MAE scores around $249,620. Ridge Regression performed marginally best with the lowest MAE (249,619.64), while Linear Regression was the least stable with the highest standard deviation (3055.23). Lasso performed between the two. The nearly identical scores across all three models suggest that regularization had minimal impact at default settings, likely because the features are already well-scaled. No signs of significant overfitting or underfitting were observed given the consistent std values across all models.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [61]:
# Feature Engineering - 3 new features using available columns
df_cleaned["living_area_ratio"] = df_cleaned["calculatedfinishedsquarefeet"] / df_cleaned["lotsizesquarefeet"]
df_cleaned["room_density"] = (df_cleaned["bedroomcnt"] + df_cleaned["bathroomcnt"]) / df_cleaned["calculatedfinishedsquarefeet"]
df_cleaned["age"] = 2025 - df_cleaned["yearbuilt"]

# Redefine X with new features
X = df_cleaned.drop(columns=["taxvaluedollarcnt"])
y = df_cleaned["taxvaluedollarcnt"]

# New train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Rescale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Re-run the 3 models
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train,
                             scoring="neg_mean_absolute_error", cv=cv)
    mae_scores = -scores
    print(f"{name}: MAE = {mae_scores.mean():.2f} +/- {mae_scores.std():.2f}")

Linear Regression: MAE = 244100.87 +/- 3405.30
Ridge Regression: MAE = 244093.97 +/- 3406.59


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.640e+14, tolerance: 1.043e+12
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.070e+14, tolerance: 9.744e+11
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the featu

Lasso Regression: MAE = 244098.91 +/- 3405.86


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.058e+14, tolerance: 1.071e+12
  model = cd_fast.enet_coordinate_descent(


In [62]:
print(df_cleaned.columns.tolist())

['parcelid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'fips', 'fullbathcnt', 'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet', 'propertylandusetypeid', 'rawcensustractandblock', 'regionidcity', 'regionidcounty', 'regionidzip', 'roomcnt', 'unitcnt', 'yearbuilt', 'assessmentyear', 'censustractandblock', 'taxvaluedollarcnt', 'living_area_ratio', 'room_density', 'age']


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




Adding three engineered features such as living area ratio, room density, and property age resulted in a modest improvement across all three models, reducing MAE by roughly $5,500. Ridge again performed best (244,093.97), followed by Lasso (244,098.91) and Linear Regression (244,100.87). The Lasso convergence warnings suggest it needs more iterations to fully optimize with the new features, but the results are still valid. The living area ratio and age features likely contributed most, as they capture property value drivers that raw square footage alone doesn't reflect.



### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [63]:
from sklearn.feature_selection import SelectKBest, f_regression


selector = SelectKBest(score_func=f_regression, k=10)
selector.fit(X_train_scaled, y_train)

feature_mask = selector.get_support()
selected_features = X.columns[feature_mask].tolist()
print("Selected features:", selected_features)

X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train_selected, y_train,
                             scoring="neg_mean_absolute_error", cv=cv)
    mae_scores = -scores
    print(f"{name}: MAE = {mae_scores.mean():.2f} +/- {mae_scores.std():.2f}")

Selected features: ['bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'fullbathcnt', 'longitude', 'living_area_ratio', 'room_density']
Linear Regression: MAE = 250891.52 +/- 3966.80
Ridge Regression: MAE = 250889.18 +/- 3966.59


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.222e+14, tolerance: 1.043e+12
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.618e+14, tolerance: 9.744e+11
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the featu

Lasso Regression: MAE = 250891.30 +/- 3966.79


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.658e+14, tolerance: 9.882e+11
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.633e+14, tolerance: 1.071e+12
  model = cd_fast.enet_coordinate_descent(


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


Feature selection using SelectKBest (k=10) did not improve performance, MAE increased slightly to ~250,891 compared to ~244,100 with all features. This suggests that the removed features, while less correlated with the target individually, still contributed useful information in combination. Ridge again performed marginally best (250,889.18). Both engineered features living_area_ratio and room_density were retained by the selector, confirming they carry predictive value. The results suggest that for this dataset, using all features is preferable to reducing them.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [64]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np


param_grid_lr = {'fit_intercept': [True, False]}
lr = GridSearchCV(LinearRegression(), param_grid_lr, cv=5, scoring='neg_mean_absolute_error')
lr.fit(X_train_scaled, y_train)
print(f"Linear Regression best params: {lr.best_params_}")
print(f"Linear Regression MAE: {-lr.best_score_:.2f}")


param_grid_ridge = {'alpha': [0.01, 0.1, 1, 10, 100, 1000]}
ridge = RandomizedSearchCV(Ridge(), param_grid_ridge, cv=5, scoring='neg_mean_absolute_error', random_state=42)
ridge.fit(X_train_scaled, y_train)
print(f"\nRidge best params: {ridge.best_params_}")
print(f"Ridge MAE: {-ridge.best_score_:.2f}")


param_grid_lasso = {'alpha': [0.01, 0.1, 1, 10, 100, 1000], 'max_iter': [5000, 10000]}
lasso = RandomizedSearchCV(Lasso(), param_grid_lasso, cv=5, scoring='neg_mean_absolute_error', random_state=42)
lasso.fit(X_train_scaled, y_train)
print(f"\nLasso best params: {lasso.best_params_}")
print(f"Lasso MAE: {-lasso.best_score_:.2f}")

Linear Regression best params: {'fit_intercept': True}
Linear Regression MAE: 244081.35


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



Ridge best params: {'alpha': 1000}
Ridge MAE: 242518.58


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.092e+14, tolerance: 1.082e+12
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.665e+14, tolerance: 1.041e+12
  model = cd_fast.enet_coordinate_descent(
/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the featu


Lasso best params: {'max_iter': 5000, 'alpha': 1000}
Lasso MAE: 243100.69


/workspaces/Module-3-Assignments/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.941e+14, tolerance: 1.057e+12
  model = cd_fast.enet_coordinate_descent(


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


For tuning, RandomizedSearchCV was used across all three models. Linear Regression had minimal tuning options but confirmed fit_intercept=True as optimal. Ridge benefited most from tuning, with alpha=1000 reducing MAE to 242,518 — the best result across all parts. Lasso also improved with alpha=1000 and max_iter=5000, though convergence warnings persisted, suggesting the dataset may not be well-suited for L1 regularization at this scale. Higher alpha values worked best for both Ridge and Lasso, indicating that stronger regularization helps reduce overfitting on this noisy housing dataset.

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [66]:
from sklearn.metrics import mean_absolute_error


final_model = Ridge(alpha=1000)


cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
scores = cross_val_score(final_model, X_train_scaled, y_train,
                         scoring="neg_mean_absolute_error", cv=cv)
mae_scores = -scores
print(f"CV MAE: {mae_scores.mean():.2f} +/- {mae_scores.std():.2f}")


final_model.fit(X_train_scaled, y_train)
test_preds = final_model.predict(X_test_scaled)
test_mae = mean_absolute_error(y_test, test_preds)
print(f"Test MAE: {test_mae:.2f}")

CV MAE: 242523.39 +/- 3489.85
Test MAE: 247282.96


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

Model Selection: Ridge Regression with alpha=1000 was selected as the final model. It consistently achieved the lowest MAE across all parts of the milestone, with a final CV MAE of 242,523 (+/- 3,489) and a test MAE of 247,282. The small gap between CV and test scores indicates the model generalizes well without significant overfitting. Ridge was chosen over Lasso due to its stability and lack of convergence issues.
Revisiting an Early Decision: During preprocessing, columns with more than 50% missing values were dropped and remaining nulls were removed. This was done to ensure clean input data for modeling. In hindsight, imputing some of these missing values rather than dropping them could have retained more data and potentially improved model performance.
Lessons Learned: This dataset is noisy and difficult to model with linear methods alone, even the best model has a MAE of ~$247k on property values. Engineered features like living_area_ratio and age provided modest improvements. Given more time, trying tree-based models like Random Forest or Gradient Boosting would likely yield significantly better results.